# Fairness in an HR shortlist

**Learning session | 2 hours | Python 3.12 | CPU laptop**

We will trace two problems in a hiring workflow: past decisions can carry bias into a model, and hiring outcomes are observed only for shortlisted applicants. We will use Fairlearn to measure group differences, then decide what those numbers can and cannot tell us.

> **All data in this notebook are simulated.** The people, group labels, scores, suitability labels, shortlist decisions, and hiring outcomes are fictional. This exercise is not evidence about real applicants or a hiring process.

Run cells from top to bottom. At each **PAUSE**, write a proposal before moving on. The saved output blocks below are snapshots from the committed seed-20261020 reports; running the cells refreshes them. Related reports in results/precomputed/ are available for deeper discussion.

## Session map

1. Historical decisions carry a score penalty.
2. Unobserved outcomes are not rejections.
3. Fairlearn measures who reaches the shortlist.
4. Recall depends on evidence beyond observed hiring outcomes.

## Load the fictional cohort

The fixed-seed cohort has 8 fictional vacancies and 120 applications per vacancy. Group fields are assigned explicitly by the generator; no attribute is inferred from a CV. The historical score has visible group penalties and access-proxy bonuses.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from fairlearn.metrics import MetricFrame, selection_rate
from sklearn.metrics import recall_score

# Find the repo whether Jupyter starts here or at the repository root.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'requirements.lock').exists())
FAIRLEARN_DIR = REPO_ROOT / '04-fairlearn'
DATA_DIR = FAIRLEARN_DIR / 'synthetic_hr_demo'
OUTPUT_DIR = FAIRLEARN_DIR / 'results' / 'precomputed'

applications = pd.read_csv(DATA_DIR / 'synthetic_historical_hr.csv')
metadata = json.loads((DATA_DIR / 'metadata.json').read_text(encoding='utf-8'))
print(f"Loaded {len(applications):,} fictional applications across {metadata['vacancy_count']} vacancies.")
print('Seed:', metadata['seed'])
print('Synthetic suitability rule:', metadata['suitability_definition'])

Loaded 960 fictional applications across 8 vacancies.
Seed: 20261020
Synthetic suitability rule: skill_match >= 62, relevant_years >= 3, and required_certificate == 1

# Episode 1: Historical decisions carry a score penalty (25 min)

### Do the obvious thing
Rank each vacancy by its historical score and keep the top five. A model trained to reproduce those decisions may appear to learn what past teams preferred.

### See the problem
The historical score adds explicit group penalties and bonuses for access proxies. These columns make the injected bias visible.

### PAUSE - 3 minutes: what would you do?
Before training on old shortlist decisions, which score components or decision records would you inspect? Who should decide whether each feature belongs? Write your answer before continuing.

In [ ]:
score_columns = [
    'gender_score_effect', 'ethnicity_score_effect',
    'economic_class_score_effect', 'geography_score_effect',
    'proxy_score_effect',
]
score_effects = applications.groupby('gender')[score_columns].mean().round(2)
display(score_effects)

baseline = json.loads((DATA_DIR / 'qualification_baseline_run.json').read_text(encoding='utf-8'))
historical = json.loads((DATA_DIR / 'biased_historical_run.json').read_text(encoding='utf-8'))
comparison = json.loads((OUTPUT_DIR / 'comparison_gender.json').read_text(encoding='utf-8'))
comparison_rows = [
    {'group': group['group'], 'count': group['count'],
     'baseline_selection_rate': group['baseline_selection_rate'],
     'historical_selection_rate': group['biased_selection_rate'],
     'selection_rate_change': group['selection_rate_change'],
     'recall_change': group['recall_change']}
    for vacancy in comparison['per_jd'] for group in vacancy['groups']
]
comparison_summary = pd.DataFrame(comparison_rows).groupby('group').agg(
    applications=('count', 'sum'),
    baseline_selection_rate=('baseline_selection_rate', 'mean'),
    historical_selection_rate=('historical_selection_rate', 'mean'),
    selection_rate_change=('selection_rate_change', 'mean'),
    recall_change=('recall_change', 'mean'),
).reset_index()
for column in ['baseline_selection_rate', 'historical_selection_rate', 'selection_rate_change', 'recall_change']:
    comparison_summary[column] = 100 * comparison_summary[column]
display(comparison_summary.round(2))

                 gender_score_effect  ethnicity_score_effect  economic_class_score_effect  geography_score_effect  proxy_score_effect
gender

Man                               0.00                   -2.46                        -4.17                   -2.88                1.78
Nonbinary                        -5.00                   -3.13                        -3.93                   -2.47                2.20
Woman                            -5.00                   -2.53                        -4.15                   -2.66                1.93

       group  applications  baseline_selection_rate  historical_selection_rate  selection_rate_change  recall_change
0        Man           464                     3.89                       5.86                  1.97          6.76
1  Nonbinary            15                     0.00                       0.00                  0.00          0.00
2      Woman           481                     4.60                       2.74                 -1.86         -6.24

### The fix and the human decision
Compare the historical ranking with the qualification-only baseline on the same applicants. A hiring specialist must decide which criteria are job-related and which need removal, investigation, or better evidence.

### Check it
The table holds the candidate pool fixed while changing the score recipe. The shock is deliberately simulated, so this shows that the audit can capture an injected change; it does not establish that real hiring is biased.

**Takeaway:** A model can reproduce past decisions faithfully while reproducing their bias too.

# Episode 2: No shortlist means no observed hiring outcome (25 min)

### Do the obvious thing
A tempting shortcut is to mark every applicant outside the historical shortlist as not hired, then use that as a complete training label.

### See the problem
Hiring outcomes are observed only for applicants who reached the shortlist. For everyone else, the outcome is unknown. A blank is not a rejection.

In [ ]:
applications['outcome_observed'] = applications['historical_hired'].notna()
coverage = applications.groupby('historical_shortlisted').agg(
    applications=('candidate_id', 'size'),
    outcomes_observed=('outcome_observed', 'sum'),
    observed_hires=('historical_hired', 'sum'),
).rename(index={0: 'not shortlisted', 1: 'shortlisted'})
display(coverage)

naive_rate = applications['historical_hired'].fillna(0).mean()
observed_only_rate = applications['historical_hired'].mean()
print(f'Naive rate after treating missing as no hire: {naive_rate:.1%}')
print(f'Observed hire rate among shortlisted applicants: {observed_only_rate:.1%}')

                 applications  outcomes_observed  observed_hires
historical_shortlisted

not shortlisted           920                  0               0
shortlisted                40                 40              13

Naive rate after treating missing as no hire: 1.4%
Observed hire rate among shortlisted applicants: 32.5%

### PAUSE - 3 minutes: what would you do?
How would you evaluate or train a model when outcomes exist only for people an earlier process selected? What evidence could tell you how excluded applicants would have performed? Write your proposal before continuing.

### The fix and the human decision
Keep unobserved outcomes missing. Report who was eligible for outcome observation and which groups are absent from that labeled subset. A person must decide whether to collect independent judgments, run a prospective review, or use a method designed for selective labels. The observed subset alone cannot answer how excluded candidates would have performed.

In [ ]:
outcome_coverage_by_gender = applications.groupby('gender').agg(
    applications=('candidate_id', 'size'),
    observed_outcomes=('outcome_observed', 'sum'),
)
outcome_coverage_by_gender['coverage_percent'] = (
    100 * outcome_coverage_by_gender['observed_outcomes'] / outcome_coverage_by_gender['applications']
)
display(outcome_coverage_by_gender.round(2))
print('Recorded coverage:', metadata['historical_outcome_observation'])

           applications  observed_outcomes  coverage_percent
gender

Man                 464                 27              5.82
Nonbinary             15                  0              0.00
Woman                481                 13              2.70

Recorded coverage: {'observed_only_for': 'historical_shortlisted == 1', 'observed_count': 40, 'missing_count': 920, 'meaning': 'historical_hired is blank for applicants outside the historical shortlist; blank means outcome not observed, not not hired'}

### Check it
The fixed-seed cohort has **40 observed outcomes and 920 missing outcomes**. The separate suitable field is a complete synthetic teaching label from a fabricated rule; it is not observed hiring history or validated ground truth.

**Takeaway:** Missing outcomes caused by an earlier selection step cannot be treated as negative outcomes.

# Episode 3: Fairlearn measures who reaches the shortlist (30 min)

### Do the obvious thing
Compare top-five selection rates by gender for the qualification baseline and simulated historical score.

### See the problem
A shortlist has only five places per vacancy. One candidate crossing the cutoff can shift a small group's rate. Show group counts and vacancy-level results beside rates.

### PAUSE - 3 minutes: what would you do?
Which denominator answers who got access to shortlist review? What else would you need before calling a difference concerning? Write your answer first.

### The fix and the human decision
Use all applications in each group as the selection-rate denominator. Fairlearn's MetricFrame calculates the group metrics; the hiring team chooses which comparison and follow-up fit the decision context.

In [ ]:
def recall_when_defined(y_true, y_pred):
    # Recall has no denominator when a group has no suitable applications.
    if np.sum(y_true) == 0:
        return np.nan
    return recall_score(y_true, y_pred, zero_division=0)

def audit_run(run, group_column):
    records = []
    for entry in run['results']:
        ranked = pd.DataFrame(entry['ranking'])
        frame = ranked.merge(
            applications[['jd_id', 'candidate_id', group_column, 'suitable']],
            on=['jd_id', 'candidate_id'], validate='one_to_one',
        )
        measured = MetricFrame(
            metrics={'selection_rate': selection_rate, 'recall': recall_when_defined},
            y_true=frame['suitable'].astype(int),
            y_pred=frame['shortlisted'].astype(int),
            sensitive_features=frame[group_column],
        )
        counts = frame[group_column].value_counts()
        for group, values in measured.by_group.iterrows():
            records.append({
                'jd_id': entry['jd_id'], 'group': group,
                'count': int(counts[group]),
                'sufficient_sample': int(counts[group]) >= 5,
                'selection_rate': float(values['selection_rate']),
                'recall': None if pd.isna(values['recall']) else float(values['recall']),
            })
    return pd.DataFrame(records)

baseline_gender = audit_run(baseline, 'gender').assign(decision='qualification baseline')
historical_gender = audit_run(historical, 'gender').assign(decision='simulated historical')
gender_audit = pd.concat([baseline_gender, historical_gender], ignore_index=True)

gender_summary = gender_audit.groupby(['decision', 'group']).agg(
    applications=('count', 'sum'),
    mean_vacancy_selection_rate=('selection_rate', 'mean'),
    mean_vacancy_recall=('recall', 'mean'),
    vacancies_with_small_group=('sufficient_sample', lambda values: int((~values).sum())),
).reset_index()
display(gender_summary.round(3))

saved = json.loads((OUTPUT_DIR / 'comparison_gender.json').read_text(encoding='utf-8'))
print('Saved comparison contains', len(saved['per_jd']), 'vacancy-level results.')


                 decision      group  applications  mean_vacancy_selection_rate  mean_vacancy_recall  vacancies_with_small_group
0  qualification baseline        Man           464                        0.039              0.155                          0
1  qualification baseline  Nonbinary            15                        0.000              0.000                          6
2  qualification baseline      Woman           481                        0.046              0.162                          0
3  simulated historical        Man           464                        0.059              0.222                          0
4  simulated historical  Nonbinary            15                        0.000              0.000                          6
5  simulated historical      Woman           481                        0.027              0.100                          0

Saved comparison contains 8 vacancy-level results.

### Check it
Selection rate uses every applicant in that group and vacancy as its denominator. Counts stay beside rates, and cells below five are flagged as thin evidence, not a statistical test. Recall uses synthetic suitability labels, not observed hiring outcomes.

**Takeaway:** A group metric describes a defined population and decision; it does not explain why a gap exists.

# Episode 4: Recall needs evidence beyond observed hiring outcomes (30 min)

### Do the obvious thing
Read selection rates and recall as if both were based on equally complete real-world outcomes.

### See the problem
The saved recall uses 960 synthetic suitability labels. Actual hiring outcomes are observed for only 40 shortlisted applicants. Recall among all real applicants cannot be inferred from these hiring outcomes. Four separate group audits also do not reveal every intersection.

### PAUSE - 3 minutes: what would you do?
What independent evidence could estimate missed suitable applicants without pretending non-shortlisted people failed? Which intersection would you examine, and how would you protect people in small cells? Write your proposal before continuing.

### The fix and the human decision
Keep shortlist access and suitability recall as separate questions. Treat this recall as a code demonstration. Before a real evaluation, people must set an independent assessment plan, privacy controls, minimum reporting counts, an uncertainty method, and an escalation owner.

In [ ]:
dimensions = ['gender', 'ethnicity', 'economic_class', 'geography']
marginal_summaries = []
for dimension in dimensions:
    audit = audit_run(historical, dimension)
    summary = audit.groupby('group').agg(
        applications=('count', 'sum'),
        mean_vacancy_selection_rate=('selection_rate', 'mean'),
        vacancies_with_small_group=('sufficient_sample', lambda values: int((~values).sum())),
    ).reset_index()
    summary.insert(0, 'dimension', dimension)
    marginal_summaries.append(summary)
marginal_summary = pd.concat(marginal_summaries, ignore_index=True)
display(marginal_summary.round(3))

print('Saved audit reports:')
for dimension in dimensions:
    report = json.loads((OUTPUT_DIR / f'biased_audit_{dimension}.json').read_text(encoding='utf-8'))
    print(f"  {dimension}: {len(report['results'])} vacancy results")

         dimension              group  applications  mean_vacancy_selection_rate  vacancies_with_small_group
0           gender                 Man           464                        0.059                          0
1           gender           Nonbinary            15                        0.000                          6
2           gender               Woman           481                        0.027                          0
3        ethnicity         Ethnicity A           385                        0.046                          0
4        ethnicity         Ethnicity B           267                        0.055                          0
5        ethnicity         Ethnicity C           197                        0.030                          0
6        ethnicity         Ethnicity D           111                        0.007                          0
7   economic_class                High           138                        0.055                          0
8   economic_class 

Saved audit reports:
  gender: 8 vacancy results
  ethnicity: 8 vacancy results
  economic_class: 8 vacancy results
  geography: 8 vacancy results

### Check it
Each table is one marginal comparison. None combines gender, ethnicity, class, and geography into intersectional groups. Small groups make rates unstable; the small-cell flag is a prompt for careful reporting, not a statistical test.

**Takeaway:** You cannot measure who a shortlist missed without suitable outcome evidence beyond the shortlist. Passing a metric calculation does not prove the model works or that its outcomes are fair.

## Close the session
Write down one human decision owner, one missing source of evidence, and one condition that would cause the team to pause a real shortlist.

For extensions, see EXERCISES.md. Provenance and limits are in data/SOURCES.md and synthetic_hr_demo/README.md. If a calculation fails, discuss the matching saved report in results/precomputed/ without rerunning.